# 06 — Annual CHM inference and qualitative maps

This notebook generates annual canopy-height estimates on the recorded 10-m output grids, verifies georeferencing, and exports the qualitative map panels used in Fig. 11. Exploratory conformal-uncertainty cells are excluded because they are not part of the submitted manuscript.

In [ ]:
from pathlib import Path
from io import BytesIO
import json
import os
import math
import re
import secrets
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patheffects
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from PIL import Image
from pyproj import Transformer

import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.windows import Window, bounds as window_bounds, from_bounds as window_from_bounds
from rasterio.warp import reproject, transform_bounds

try:
    import requests
except Exception:
    requests = None

try:
    import contextily as ctx
except Exception:
    ctx = None

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
PRODUCT_ROOT = PROJECT / "CHM_Products_Comparison"
OUT_DIR = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Inference_Maps"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_DPI = 600
NODATA_COLOR = "#E6E6E6"
CMAP_NAME = "magma"

# Satellite-context configuration.
# Google Maps Static does not expose the acquisition date of its current
# satellite mosaic. The 2020 tag below denotes the comparison/reference year,
# not a guaranteed imagery acquisition date. A genuinely historical 2020
# Google image can be placed in the cache path printed by the preflight.
BASEMAP_REFERENCE_YEAR = 2020

# A fresh seed is generated at every kernel execution. Copy the printed seed
# here when an article panel must be reproduced exactly.
RANDOM_ROI_SEED = None
ACTIVE_RANDOM_ROI_SEED = (
    int(RANDOM_ROI_SEED)
    if RANDOM_ROI_SEED is not None
    else secrets.randbits(63)
)
RANDOM_ROI_GENERATOR = np.random.default_rng(ACTIVE_RANDOM_ROI_SEED)

# Do not silently reuse the same qualitative window on a later execution.
# Delete this JSON only when a complete reset of the visual sampling is wanted.
AVOID_PREVIOUS_RANDOM_ROIS = True
ROI_HISTORY_PATH = OUT_DIR / "00_random_roi_history.json"

MIN_GLOBAL_PRODUCT_VALID_FRACTION = 0.20
REQUIRE_LOCAL_FULL_VALID_FRACTION = 1.0
BASEMAP_MODE = "auto"  # auto | google_static | esri
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "").strip()
GOOGLE_STATIC_ZOOM = 15
ESRI_ZOOM = 17
GOOGLE_STATIC_SIZE = "640x640"
GOOGLE_STATIC_SCALE = 2
BASEMAP_CACHE = OUT_DIR / "_basemap_cache"
BASEMAP_CACHE.mkdir(parents=True, exist_ok=True)

S2_ROOTS = {
    "Ifran": Path(r"E:\CHM\Ifran_6\DATA\S2\S2_MONTHLY_IFRAN_CLEAN_V1"),
    "Maamoura": Path(r"E:\CHM\Maamoura\Data\S2_12_Bands"),
    "Agadir": Path(r"E:\CHM\Agadir\Data\S2_12_Bands"),
}

SITES = {
    "Ifran": {
        "key": "Ifran_6",
        "ecosystem": "Moderately dense",
        "year": 2020,
        "vmax": 40.0,
        "roi_metres": 700.0,
        "ours": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Dense/Ifran/Phase2/Y2020/Annual/Ifran_B4_C15_Phase2_Y2020_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
    },
    "Maamoura": {
        "key": "Maamoura",
        "ecosystem": "Low-density",
        "year": 2019,
        "vmax": 20.0,
        "roi_metres": 700.0,
        "ours": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Low_Sparsity/Maamoura/Phase2/Y2019/Annual/Maamoura_B4_C15_Phase2_Y2019_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
    },
    "Agadir": {
        "key": "Agadir",
        "ecosystem": "Sparse",
        "year": 2020,
        "vmax": 20.0,
        "roi_metres": 700.0,
        "ours": PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1/Sparse/Agadir/Phase2/Y2020/Annual/Agadir_B4_C15_Phase2_Y2020_M05-09_T4_SLIDING_2018-2025_ENSEMBLE.tif",
    },
}

MANUAL_ROI_BOUNDS = {
    "Ifran": None,
    "Maamoura": None,
    "Agadir": None,
}

PRODUCT_ORDER = ["Our Model", "Pauls Pa24", "Lang L23", "Tolan T24", "Potapov P21"]
PRODUCT_TITLES = {
    "Our Model": "Our Model",
    "Lang L23": "Lang et al. (L23)",
    "Pauls Pa24": "Pauls et al. (Pa24)",
    "Tolan T24": "Tolan et al. (T24)",
    "Potapov P21": "Potapov et al. (P21)",
}

for forest, cfg in SITES.items():
    root = PRODUCT_ROOT / cfg["key"]
    pauls_epsg = "EPSG32630" if forest == "Ifran" else "EPSG32629"
    pauls_root = (
        PRODUCT_ROOT / "_PAULS_2020_VERIFIED_V1" / forest
        / "Pauls_et_al_2024_CHM_2020_10m" / "clean" / "mosaic"
    )
    cfg["maps"] = {
        "Our Model": cfg["ours"],
        "Lang L23": root / f"ETH_Lang_2020_CHM_10m/clean/mosaic/{cfg['key']}__ETH_Lang_2020_CHM_10m__clean__EPSG32630.tif",
        "Pauls Pa24": pauls_root / f"{forest}__Pauls_et_al_2024_CHM_2020_10m__clean__{pauls_epsg}.tif",
        "Tolan T24": root / f"Meta_WRI_Tolan_2023_CHM_resampled_10m/clean/mosaic/{cfg['key']}__Meta_WRI_Tolan_2023_CHM_resampled_10m__clean__EPSG32630.tif",
        "Potapov P21": root / f"GFCH_Potapov_GLAD_2019_30m/clean/mosaic/{cfg['key']}__GFCH_Potapov_GLAD_2019_30m__clean__EPSG32630.tif",
    }

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "semibold",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

INFERENCE_SCRIPT = PROJECT / "Source" / "Project" / "phase2_c15_inference_three_forests.py"
INFERENCE_ROOT = PROJECT / "Inference_Harmonized_GEDIAnchored_NaturalP1"
PHASE1_EXPECTED_SHA256 = {
    "Ifran": "072a8735973e3511564cf3ef7907f5b33105df08895c78917fb817de6198afb1",
    "Maamoura": "d83a5493715451a61c997a66a25f361080a56e7ceade60eb886f2ae76f6bd7f4",
    "Agadir": "f72f06e33455852cbc94cd00ecb81e192f68e7f67b89de007fef4a610fa25b2e",
}

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
PYTHON = Path(
    r"C:\Users\Dell\Desktop\Article_Maroc_Agadir"
    r"\Env_Workspace_agadir\.venv310\Scripts\python.exe"
)
if not PYTHON.is_file():
    import sys
    PYTHON = Path(sys.executable)

# Reproducible execution switches. Existing maps are reused only when their
# manifest contains the current Phase-2 checkpoint hash and T4 windows.
RUN_INFERENCE = True  # regenerate maps when the manifest hash is stale
OVERWRITE_EXISTING_MAPS = False
SAVE_MONTHLY_MAPS = False
RUN_QUALITATIVE_FIGURES = True

assert INFERENCE_SCRIPT.is_file(), INFERENCE_SCRIPT
assert PYTHON.is_file(), PYTHON

print("Output directory:", OUT_DIR)
print("Inference root:", INFERENCE_ROOT)
print("Inference script:", INFERENCE_SCRIPT)
print("Python:", PYTHON)
print("Random ROI seed:", ACTIVE_RANDOM_ROI_SEED)


In [ ]:

import subprocess


def _stream_subprocess(command: list[str]) -> None:
    """Stream the child process and fail on any non-zero return code."""
    environment = dict(os.environ)
    environment["PYTHONIOENCODING"] = "utf-8"
    environment["PYTHONUTF8"] = "1"
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        env=environment,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


def _verify_inference_artifacts(forest: str) -> None:
    """Never report READY unless the requested map and its QA manifest exist."""
    if forest == "all":
        return
    forest_name = {"ifran": "Ifran", "maamoura": "Maamoura", "agadir": "Agadir"}[forest]
    output = Path(SITES[forest_name]["ours"])
    manifest = output.parent.parent / "QA" / "inference_manifest.json"
    missing = [path for path in (output, manifest) if not path.is_file()]
    if missing:
        details = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(
            f"{forest_name}: inference command finished but required artifacts are missing:\n{details}"
        )
    print(f"[VERIFIED] Phase-2 map: {output}", flush=True)
    print(f"[VERIFIED] QA manifest: {manifest}", flush=True)


def run_inference_command(forest: str, *, run: bool) -> list[str]:
    """Run Phase-2 inference, stream progress, and verify generated artifacts."""
    command = [
        str(PYTHON), "-u", str(INFERENCE_SCRIPT),
        "--forest", forest, "--preflight",
    ]
    if run:
        command.append("--run")
        if OVERWRITE_EXISTING_MAPS:
            command.append("--overwrite")
        if SAVE_MONTHLY_MAPS:
            command.append("--save-monthly")
    print("\n[COMMAND]", subprocess.list2cmdline(command), flush=True)
    _stream_subprocess(command)
    if run:
        _verify_inference_artifacts(forest)
    return command


# Global preflight checks the C15 catalogs, Phase-1/Phase-2 checkpoints,
# complete T4 sliding windows and the Sentinel-2 reference rasters.
run_inference_command("all", run=False)
print("\n[PASS] Global Phase-2 inference preflight.")


In [ ]:

run_inference_command("ifran", run=RUN_INFERENCE)
print("[READY] Ifran Phase 2 inference")


In [ ]:

run_inference_command("maamoura", run=RUN_INFERENCE)
print("[READY] Maamoura Phase 2 inference")


In [ ]:

run_inference_command("agadir", run=RUN_INFERENCE)
print("[READY] Agadir Phase 2 inference")


In [ ]:

def _same_transform(a, b, atol: float = 1e-9) -> bool:
    return bool(np.allclose(tuple(a), tuple(b), rtol=0.0, atol=atol))


def _overlap_fraction_in_reference_crs(source, reference) -> float:
    sb = transform_bounds(
        source.crs, reference.crs, *source.bounds, densify_pts=21
    )
    rb = reference.bounds
    width = max(0.0, min(sb[2], rb.right) - max(sb[0], rb.left))
    height = max(0.0, min(sb[3], rb.top) - max(sb[1], rb.bottom))
    reference_area = (rb.right - rb.left) * (rb.top - rb.bottom)
    return float(width * height / reference_area) if reference_area > 0 else 0.0


def strict_geospatial_audit() -> pd.DataFrame:
    """Validate native metadata and the exact destination-grid contract."""
    rows = []
    for forest, cfg in SITES.items():
        ours_path = Path(cfg["ours"])
        manifest_path = (
            ours_path.parent.parent / "QA" / "inference_manifest.json"
        )
        if not ours_path.is_file():
            raise FileNotFoundError(f"{forest}: missing Phase-2 map: {ours_path}")
        if not manifest_path.is_file():
            raise FileNotFoundError(
                f"{forest}: missing inference manifest: {manifest_path}"
            )

        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        phase1_checkpoint = Path(manifest.get("phase1_checkpoint", ""))
        phase2_checkpoint = Path(manifest.get("phase2_checkpoint", ""))
        if not phase1_checkpoint.is_file() or not phase2_checkpoint.is_file():
            raise FileNotFoundError(f"{forest}: manifest checkpoint missing")
        if manifest.get("phase1_sha256") != PHASE1_EXPECTED_SHA256[forest]:
            raise RuntimeError(f"{forest}: annual map has the wrong Phase 1 parent lineage")
        if file_sha256(phase1_checkpoint) != manifest.get("phase1_sha256"):
            raise RuntimeError(f"{forest}: Phase 1 checkpoint hash mismatch")
        if file_sha256(phase2_checkpoint) != manifest.get("phase2_sha256"):
            raise RuntimeError(f"{forest}: Phase 2 checkpoint hash mismatch")
        selected_registry = json.loads((PROJECT / "Source" / "Project" / "final_selected_phase2_models.json").read_text(encoding="utf-8"))["models"]
        selected = selected_registry[forest.lower()]
        if manifest.get("phase2_sha256") != selected["checkpoint_sha256"]:
            raise RuntimeError(f"{forest}: annual map Phase 2 hash differs from the final registry")
        reference_path = Path(manifest["reference_raster"])
        if not reference_path.is_file():
            raise FileNotFoundError(
                f"{forest}: missing Sentinel-2 reference: {reference_path}"
            )

        with rasterio.open(ours_path) as ours, rasterio.open(reference_path) as ref:
            if ours.crs is None or ref.crs is None:
                raise RuntimeError(f"{forest}: missing CRS on output/reference")
            if not ours.crs.is_projected or not ref.crs.is_projected:
                raise RuntimeError(f"{forest}: output/reference CRS must be projected")
            exact_grid = (
                ours.crs == ref.crs
                and ours.width == ref.width
                and ours.height == ref.height
                and _same_transform(ours.transform, ref.transform)
            )
            if not exact_grid:
                raise RuntimeError(
                    f"{forest}: Phase-2 output is not on its exact Sentinel-2 grid"
                )
            if not np.isclose(abs(ours.res[0]), abs(ours.res[1])):
                raise RuntimeError(f"{forest}: non-square output pixels: {ours.res}")
            if not np.isclose(ours.transform.b, 0.0) or not np.isclose(
                ours.transform.d, 0.0
            ):
                raise RuntimeError(f"{forest}: rotated/skewed output grid")

            reference_crs = str(ours.crs)
            reference_epsg = ours.crs.to_epsg()
            reference_resolution = abs(float(ours.res[0]))

            for product, path in cfg["maps"].items():
                path = Path(path)
                if not path.is_file():
                    rows.append({
                        "forest": forest,
                        "product": product,
                        "status": "MISSING",
                        "path": str(path),
                    })
                    continue
                with rasterio.open(path) as src:
                    if src.crs is None:
                        raise RuntimeError(f"{forest}/{product}: missing CRS")
                    if not src.crs.is_projected:
                        raise RuntimeError(
                            f"{forest}/{product}: native CRS is not projected"
                        )
                    if not np.isclose(abs(src.res[0]), abs(src.res[1])):
                        raise RuntimeError(
                            f"{forest}/{product}: non-square pixels {src.res}"
                        )
                    if not np.isclose(src.transform.b, 0.0) or not np.isclose(
                        src.transform.d, 0.0
                    ):
                        raise RuntimeError(
                            f"{forest}/{product}: rotated/skewed native grid"
                        )
                    overlap = _overlap_fraction_in_reference_crs(src, ours)
                    if overlap < 0.95:
                        raise RuntimeError(
                            f"{forest}/{product}: only {overlap:.1%} spatial "
                            "overlap after CRS transformation"
                        )
                    native_resolution = abs(float(src.res[0]))
                    expected_resolution = (
                        30.0 if product == "Potapov P21" else 10.0
                    )
                    if not np.isclose(native_resolution, expected_resolution):
                        raise RuntimeError(
                            f"{forest}/{product}: expected native "
                            f"{expected_resolution:g} m, got "
                            f"{native_resolution:g} m"
                        )
                    rows.append({
                        "forest": forest,
                        "product": product,
                        "status": "PASS",
                        "native_crs": str(src.crs),
                        "native_epsg": src.crs.to_epsg(),
                        "native_resolution_m": native_resolution,
                        "reference_crs": reference_crs,
                        "reference_epsg": reference_epsg,
                        "reference_resolution_m": reference_resolution,
                        "same_native_grid_as_ours": (
                            src.crs == ours.crs
                            and src.width == ours.width
                            and src.height == ours.height
                            and _same_transform(src.transform, ours.transform)
                        ),
                        "overlap_after_crs_transform": overlap,
                        "comparison_resampling": (
                            "nearest (preserve native 30 m cells)"
                            if product == "Potapov P21"
                            else "bilinear (continuous height surface)"
                        ),
                        "path": str(path),
                    })

    report = pd.DataFrame(rows)
    if (report["status"] != "PASS").any():
        missing = report.loc[report["status"] != "PASS", [
            "forest", "product", "status", "path"
        ]]
        raise FileNotFoundError(
            "Incomplete CHM inputs:\n" + missing.to_string(index=False)
        )
    return report


geospatial_audit = strict_geospatial_audit()
display(geospatial_audit)
audit_path = OUT_DIR / "00_geospatial_crs_grid_audit.csv"
geospatial_audit.to_csv(audit_path, index=False)
print("[PASS] CRS, native resolutions, spatial overlap and exact output grids.")
print("Audit saved:", audit_path)


In [ ]:
def discover_s2(forest: str, year: int) -> Path | None:
    root = S2_ROOTS[forest]
    if not root.is_dir():
        return None
    candidates = sorted(root.rglob("*.tif"))
    year_tokens = (f"Y{year}", f"_{year}_", str(year))
    candidates = [p for p in candidates if any(token.lower() in p.name.lower() for token in year_tokens)]
    if not candidates:
        return None
    month_priority = ("M07", "_07_", "M08", "_08_", "M06", "_06_", "M09", "_09_", "M05", "_05_")
    for token in month_priority:
        hits = [p for p in candidates if token.lower() in p.name.lower()]
        if hits:
            return hits[0]
    return candidates[0]


def raster_summary(path: Path) -> dict:
    if not path.is_file():
        return {"exists": False}
    with rasterio.open(path) as src:
        return {
            "exists": True,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "bands": src.count,
            "resolution_m": round(abs(float(src.transform.a)), 3),
            "nodata": src.nodata,
        }


preflight_rows = []
for forest, cfg in SITES.items():
    cfg["s2"] = discover_s2(forest, cfg["year"])
    for product, path in cfg["maps"].items():
        info = raster_summary(path)
        preflight_rows.append({
            "forest": forest,
            "source": product,
            "path": str(path),
            **info,
        })
    s2_info = raster_summary(cfg["s2"]) if cfg["s2"] else {"exists": False}
    preflight_rows.append({
        "forest": forest,
        "source": f"Sentinel-2 fallback {cfg['year']}",
        "path": str(cfg["s2"]) if cfg["s2"] else "NOT FOUND",
        **s2_info,
    })

preflight = pd.DataFrame(preflight_rows)
display(preflight)

missing_ours = preflight[(preflight["source"] == "Our Model") & (~preflight["exists"])]
if not missing_ours.empty:
    raise FileNotFoundError("A final Phase 2 map is missing:\n" + missing_ours[["forest", "path"]].to_string(index=False))

preflight.to_csv(OUT_DIR / "00_input_preflight.csv", index=False)
print("Preflight saved:", OUT_DIR / "00_input_preflight.csv")


for forest in SITES:
    cache_path = BASEMAP_CACHE / "GoogleMaps_2020" / f"{forest}_GoogleMaps_2020.png"
    print(f"{forest}: verified Google Maps 2020 cache -> {cache_path}")
print("Google Static API key configured:", bool(GOOGLE_MAPS_API_KEY))
print("Esri/contextily fallback available:", ctx is not None)


In [ ]:
def clean_height(array: np.ndarray, vmax_physical: float = 80.0) -> np.ndarray:
    out = np.asarray(array, dtype=np.float32)
    out[~np.isfinite(out)] = np.nan
    out[(out < 0.0) | (out > vmax_physical)] = np.nan
    return out


def reproject_band_to_grid(
    path: Path,
    dst_shape: tuple[int, int],
    dst_transform,
    dst_crs,
    band: int = 1,
    resampling: Resampling = Resampling.bilinear,
) -> np.ndarray:
    dst = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(path) as src:
        reproject(
            source=rasterio.band(src, band),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            src_nodata=src.nodata,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            dst_nodata=np.nan,
            resampling=resampling,
        )
    return dst


def rgb_band_indices(src) -> tuple[int, int, int]:
    descriptions = [str(value or "").upper().replace("_", "") for value in src.descriptions]
    aliases = {
        "R": ("B04", "B4", "RED"),
        "G": ("B03", "B3", "GREEN"),
        "B": ("B02", "B2", "BLUE"),
    }
    found = {}
    for color, names in aliases.items():
        for index, description in enumerate(descriptions, start=1):
            if description in names or any(name in description for name in names):
                found[color] = index
                break
    if len(found) == 3:
        return found["R"], found["G"], found["B"]
    if src.count >= 4:
        return 4, 3, 2
    if src.count >= 3:
        return 1, 2, 3
    raise RuntimeError(f"Cannot infer RGB bands from {src.name}; count={src.count}, descriptions={src.descriptions}")


def stretch_rgb(rgb: np.ndarray, mask: np.ndarray | None = None) -> np.ndarray:
    result = np.zeros_like(rgb, dtype=np.float32)
    valid_mask = np.ones(rgb.shape[1:], dtype=bool) if mask is None else mask.copy()
    valid_mask &= np.all(np.isfinite(rgb), axis=0)
    for channel in range(3):
        values = rgb[channel][valid_mask]
        if values.size == 0:
            continue
        low, high = np.nanpercentile(values, [2.0, 98.0])
        if not np.isfinite(high) or high <= low:
            high = low + 1.0
        result[channel] = np.clip((rgb[channel] - low) / (high - low), 0.0, 1.0)
    return np.moveaxis(result, 0, -1)


def overview_stack(forest: str, max_size: int = 640) -> dict:
    cfg = SITES[forest]
    with rasterio.open(cfg["ours"]) as ref:
        scale = max(ref.width, ref.height) / max_size
        width = max(1, int(round(ref.width / max(scale, 1.0))))
        height = max(1, int(round(ref.height / max(scale, 1.0))))
        transform = from_bounds(*ref.bounds, width, height)
        crs = ref.crs

    arrays = {}
    for product, path in cfg["maps"].items():
        if not path.is_file():
            continue
        product_resampling = (
            Resampling.nearest if product == "Potapov P21"
            else Resampling.bilinear
        )
        arrays[product] = clean_height(
            reproject_band_to_grid(
                path, (height, width), transform, crs,
                resampling=product_resampling,
            )
        )
    return {
        "arrays": arrays,
        "transform": transform,
        "crs": crs,
        "shape": (height, width),
    }


def _google_cache_path(forest: str, bounds=None) -> Path:
    """Cache a satellite image by forest *and exact random ROI bounds*."""
    if bounds is None:
        suffix = "legacy"
    else:
        payload = ",".join(f"{float(value):.3f}" for value in bounds)
        suffix = hashlib.sha1(payload.encode("ascii")).hexdigest()[:12]
    return (
        BASEMAP_CACHE / "GoogleMaps_2020"
        / f"{forest}_GoogleMaps_2020_{suffix}.png"
    )


def _read_cached_rgb(path: Path) -> np.ndarray | None:
    if not path.is_file():
        return None
    image = np.asarray(Image.open(path).convert("RGB"))
    return (
        image
        if image.ndim == 3 and image.shape[2] == 3
        else None
    )


def _fetch_google_static(
    forest: str,
    bounds,
    crs,
) -> tuple[np.ndarray | None, str]:
    cache = _google_cache_path(forest, bounds)
    cached = _read_cached_rgb(cache)
    if cached is not None:
        return cached, "Google Maps (verified 2020 cache)"
    if BASEMAP_MODE not in {"auto", "google_static"}:
        return None, ""
    if not GOOGLE_MAPS_API_KEY or requests is None:
        return None, ""

    left, bottom, right, top = bounds
    transformer = Transformer.from_crs(
        crs, "EPSG:4326", always_xy=True
    )
    lon, lat = transformer.transform(
        (left + right) / 2.0,
        (bottom + top) / 2.0,
    )
    response = requests.get(
        "https://maps.googleapis.com/maps/api/staticmap",
        params={
            "center": f"{lat:.7f},{lon:.7f}",
            "zoom": str(GOOGLE_STATIC_ZOOM),
            "size": GOOGLE_STATIC_SIZE,
            "scale": str(GOOGLE_STATIC_SCALE),
            "maptype": "satellite",
            "format": "png",
            "key": GOOGLE_MAPS_API_KEY,
        },
        timeout=45,
    )
    response.raise_for_status()
    content_type = response.headers.get("Content-Type", "")
    if "image" not in content_type.lower():
        raise RuntimeError(
            "Google Static Maps returned non-image content"
        )
    image = np.asarray(
        Image.open(BytesIO(response.content)).convert("RGB")
    )
    cache.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(image).save(cache)
    return (
        image,
        "Google Maps Static (current mosaic; reference year 2020)",
    )


def _fetch_esri(bounds, crs) -> tuple[np.ndarray | None, str]:
    if (
        BASEMAP_MODE not in {"auto", "esri", "google_static"}
        or ctx is None
    ):
        return None, ""
    left, bottom, right, top = bounds
    transformer = Transformer.from_crs(
        crs, "EPSG:3857", always_xy=True
    )
    left_3857, bottom_3857, right_3857, top_3857 = (
        transformer.transform_bounds(left, bottom, right, top)
    )
    image, tile_extent = ctx.bounds2img(
        left_3857,
        bottom_3857,
        right_3857,
        top_3857,
        zoom=int(ESRI_ZOOM),
        source=ctx.providers.Esri.WorldImagery,
        ll=False,
    )
    image = np.asarray(image)[..., :3]

    # contextily returns complete Web-Mercator tiles, whose extent is usually
    # larger than the requested ROI. Crop geospatially to the requested bounds
    # instead of applying a non-georeferenced centre crop.
    tile_left, tile_right, tile_bottom, tile_top = map(float, tile_extent)
    rows, cols = image.shape[:2]
    xres = (tile_right - tile_left) / cols
    yres = (tile_top - tile_bottom) / rows
    col0 = int(np.floor((left_3857 - tile_left) / xres))
    col1 = int(np.ceil((right_3857 - tile_left) / xres))
    row0 = int(np.floor((tile_top - top_3857) / yres))
    row1 = int(np.ceil((tile_top - bottom_3857) / yres))
    col0, col1 = max(0, col0), min(cols, col1)
    row0, row1 = max(0, row0), min(rows, row1)
    if col1 <= col0 or row1 <= row0:
        raise RuntimeError("Esri crop does not intersect the requested ROI")
    return image[row0:row1, col0:col1], "Esri World Imagery"


def load_satellite_context(
    forest: str,
    bounds,
    crs,
    sentinel_rgb=None,
):
    """Load Google cache/API, Esri fallback, then Sentinel-2 fallback."""
    try:
        image, source = _fetch_google_static(
            forest, bounds, crs
        )
        if image is not None:
            return image, source
    except Exception as exc:
        warnings.warn(
            f"{forest}: Google Maps unavailable: {exc}"
        )
    try:
        image, source = _fetch_esri(bounds, crs)
        if image is not None:
            return image, source
    except Exception as exc:
        warnings.warn(
            f"{forest}: Esri World Imagery unavailable: {exc}"
        )
    if sentinel_rgb is not None:
        return (
            sentinel_rgb,
            f"Sentinel-2 RGB ({SITES[forest]['year']}) fallback",
        )
    return None, "Satellite context unavailable"


def load_roi_history() -> dict:
    """Load persistent ROI history; malformed files fail loudly."""
    if not ROI_HISTORY_PATH.is_file():
        return {forest: [] for forest in SITES}
    payload = json.loads(ROI_HISTORY_PATH.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise RuntimeError(f"Invalid ROI history: {ROI_HISTORY_PATH}")
    return {
        forest: list(payload.get(forest, []))
        for forest in SITES
    }


def save_roi_history(history: dict) -> None:
    ROI_HISTORY_PATH.write_text(
        json.dumps(history, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )


def roi_candidate_key(row: int, col: int, size: int) -> str:
    return f"r{int(row)}_c{int(col)}_s{int(size)}"


def automatic_roi(forest: str) -> tuple[float, float, float, float]:
    """Randomly select a fully valid local window.

    Products with negligible forest-wide valid coverage are not allowed to
    eliminate the useful support of all other maps. This notably classifies
    Potapov P21 as non-evaluable in Agadir.
    """
    cfg = SITES[forest]
    if MANUAL_ROI_BOUNDS[forest] is not None:
        return tuple(float(v) for v in MANUAL_ROI_BOUNDS[forest])

    overview = overview_stack(forest)
    arrays = overview["arrays"]
    transform = overview["transform"]
    pixel_size = abs(float(transform.a))
    win_pixels = max(10, int(round(cfg["roi_metres"] / pixel_size)))
    win_pixels = min(
        win_pixels, overview["shape"][0], overview["shape"][1]
    )
    step = max(1, win_pixels // 5)

    global_valid_fraction = {
        name: float(np.isfinite(array).mean())
        for name, array in arrays.items()
    }
    eligible = [
        name for name in PRODUCT_ORDER
        if name in arrays
        and (
            name == "Our Model"
            or global_valid_fraction[name]
            >= MIN_GLOBAL_PRODUCT_VALID_FRACTION
        )
    ]
    excluded = [
        name for name in PRODUCT_ORDER
        if name in arrays and name not in eligible
    ]
    cfg["roi_eligible_products"] = eligible
    cfg["roi_excluded_products"] = excluded
    cfg["global_valid_fraction"] = global_valid_fraction

    common = np.ones(overview["shape"], dtype=bool)
    for name in eligible:
        common &= np.isfinite(arrays[name])

    full_valid_candidates = []
    best_candidates = []
    best_coverage = -1.0
    for row in range(
        0, overview["shape"][0] - win_pixels + 1, step
    ):
        for col in range(
            0, overview["shape"][1] - win_pixels + 1, step
        ):
            region = np.s_[
                row:row + win_pixels,
                col:col + win_pixels,
            ]
            valid = common[region]
            coverage = float(valid.mean())
            ours_values = arrays["Our Model"][region][valid]
            if ours_values.size == 0:
                continue
            spread = float(
                np.nanpercentile(ours_values, 95)
                - np.nanpercentile(ours_values, 5)
            )
            candidate = (row, col, coverage, spread)
            if coverage >= REQUIRE_LOCAL_FULL_VALID_FRACTION:
                full_valid_candidates.append(candidate)
            if coverage > best_coverage + 1e-12:
                best_coverage = coverage
                best_candidates = [candidate]
            elif np.isclose(coverage, best_coverage):
                best_candidates.append(candidate)

    if full_valid_candidates:
        # Keep heterogeneous scenes, then randomly choose among them.
        spreads = np.array(
            [candidate[3] for candidate in full_valid_candidates],
            dtype=float,
        )
        spread_cut = float(np.nanpercentile(spreads, 35))
        candidate_pool = [
            candidate for candidate in full_valid_candidates
            if candidate[3] >= spread_cut
        ]
        selection_status = "FULL_VALID_RANDOM"
    elif best_candidates:
        candidate_pool = list(best_candidates)
        selection_status = "BEST_AVAILABLE_RANDOM"
        warnings.warn(
            f"{forest}: no 100% valid {win_pixels}x{win_pixels} window "
            f"for {eligible}; using the best available coverage."
        )
    else:
        raise RuntimeError(f"{forest}: no valid random ROI candidate")

    history = load_roi_history()
    used_keys = {
        str(item.get("candidate_key"))
        for item in history.get(forest, [])
        if isinstance(item, dict)
    }
    unseen_pool = [
        candidate for candidate in candidate_pool
        if roi_candidate_key(candidate[0], candidate[1], win_pixels)
        not in used_keys
    ]
    if AVOID_PREVIOUS_RANDOM_ROIS and unseen_pool:
        draw_pool = unseen_pool
    elif AVOID_PREVIOUS_RANDOM_ROIS and not unseen_pool:
        # This should be rare. Once every eligible window has been used,
        # restart this forest's history instead of failing the workflow.
        history[forest] = []
        draw_pool = candidate_pool
        selection_status += "_HISTORY_RESET"
    else:
        draw_pool = candidate_pool

    selected = draw_pool[
        int(RANDOM_ROI_GENERATOR.integers(0, len(draw_pool)))
    ]
    row, col, selected_coverage, selected_spread = selected
    candidate_key = roi_candidate_key(row, col, win_pixels)
    history.setdefault(forest, []).append(
        {
            "candidate_key": candidate_key,
            "row": int(row),
            "col": int(col),
            "size_pixels": int(win_pixels),
            "seed": int(ACTIVE_RANDOM_ROI_SEED),
            "coverage": float(selected_coverage),
            "height_spread_m": float(selected_spread),
        }
    )
    save_roi_history(history)

    cfg["roi_selection_status"] = selection_status
    cfg["roi_candidate_key"] = candidate_key
    cfg["roi_history_count"] = len(history[forest])
    cfg["roi_overview_valid_fraction"] = float(selected_coverage)
    window = Window(col, row, win_pixels, win_pixels)
    return tuple(
        float(value) for value in window_bounds(window, transform)
    )


def load_forest_roi(forest: str) -> dict:
    cfg = SITES[forest]
    roi_bounds = automatic_roi(forest)

    with rasterio.open(cfg["ours"]) as ref:
        window = window_from_bounds(*roi_bounds, transform=ref.transform)
        window = window.round_offsets().round_lengths()
        window = Window(
            max(0, window.col_off),
            max(0, window.row_off),
            min(window.width, ref.width - max(0, window.col_off)),
            min(window.height, ref.height - max(0, window.row_off)),
        )
        height, width = int(window.height), int(window.width)
        transform = ref.window_transform(window)
        crs = ref.crs
        exact_bounds = tuple(float(value) for value in window_bounds(window, ref.transform))

    maps = {}
    for product, path in cfg["maps"].items():
        if path.is_file():
            product_resampling = (
                Resampling.nearest if product == "Potapov P21"
                else Resampling.bilinear
            )
            maps[product] = clean_height(
                reproject_band_to_grid(
                    path, (height, width), transform, crs,
                    resampling=product_resampling,
                )
            )

    exact_valid_fraction = {
        name: float(np.isfinite(array).mean())
        for name, array in maps.items()
    }

    # Publication panels must not contain partial NoData holes. Validate the
    # exact, post-reprojection arrays (not canopy-height values): a valid 0 m
    # prediction is retained, while NaN/NoData rejects the whole ROI. Products
    # excluded forest-wide because they are unavailable (notably P21 in
    # Agadir) remain eligible for the explicit "Not available" panel.
    required_local_products = [
        name for name in cfg.get("roi_eligible_products", ["Our Model"])
        if name in maps
    ]
    invalid_required = {
        name: exact_valid_fraction.get(name, 0.0)
        for name in required_local_products
        if exact_valid_fraction.get(name, 0.0) < 1.0
    }
    if invalid_required:
        details = ", ".join(
            f"{name}={fraction:.2%}"
            for name, fraction in invalid_required.items()
        )
        raise RuntimeError(
            f"{forest}: selected ROI contains post-reprojection NoData "
            f"pixels ({details}). Draw another random zone."
        )
    locally_invalid = [
        name for name, fraction in exact_valid_fraction.items()
        if name not in required_local_products and fraction < 1.0
    ]
    # Keep every available CHM in the qualitative figure, including
    # Potapov. A product with incomplete local coverage remains visible and
    # only its invalid pixels are rendered with NODATA_COLOR. The audit table
    # records the exact valid fraction so display never implies full coverage.
    cfg["exact_valid_fraction_before_filter"] = exact_valid_fraction
    cfg["locally_invalid_products"] = locally_invalid

    if cfg["s2"] is None or not cfg["s2"].is_file():
        rgb = None
    else:
        with rasterio.open(cfg["s2"]) as s2_src:
            red, green, blue = rgb_band_indices(s2_src)
        raw_rgb = np.stack([
            reproject_band_to_grid(cfg["s2"], (height, width), transform, crs, band=index)
            for index in (red, green, blue)
        ])
        rgb = raw_rgb

    common = np.ones((height, width), dtype=bool)
    for product in PRODUCT_ORDER:
        if product in maps:
            common &= np.isfinite(maps[product])

    if common.mean() < 0.15:
        warnings.warn(
            f"{forest}: strict common coverage is only {common.mean():.1%}; "
            "falling back to the valid support of Our Model."
        )
        common = np.isfinite(maps["Our Model"])

    rgb_display = None if rgb is None else stretch_rgb(rgb, common)
    basemap, basemap_source = load_satellite_context(
        forest, exact_bounds, crs, sentinel_rgb=rgb_display
    )
    return {
        "forest": forest,
        "bounds": exact_bounds,
        "transform": transform,
        "crs": crs,
        "maps": maps,
        "rgb": rgb_display,
        "basemap": basemap,
        "basemap_source": basemap_source,
        "common_mask": common,
        "coverage": float(common.mean()),
        "pixel_size": abs(float(transform.a)),
    }


MAX_RANDOM_ROI_ATTEMPTS = 100


def _load_all_forest_rois_with_retry(reason: str) -> dict:
    """Reject post-reprojection NoData ROIs and draw another candidate."""
    last_error = None
    for attempt in range(1, MAX_RANDOM_ROI_ATTEMPTS + 1):
        try:
            selected = {forest: load_forest_roi(forest) for forest in SITES}
            if attempt > 1:
                print(
                    f"[ROI ACCEPTED] reason={reason} | attempt={attempt}/"
                    f"{MAX_RANDOM_ROI_ATTEMPTS}",
                    flush=True,
                )
            return selected
        except RuntimeError as error:
            if not any(marker in str(error) for marker in (
                "selected ROI is not fully valid for Our Model",
                "selected ROI contains post-reprojection NoData pixels",
            )):
                raise
            last_error = error
            print(
                f"[ROI REJECTED] reason={reason} | attempt={attempt}/"
                f"{MAX_RANDOM_ROI_ATTEMPTS} | {error}",
                flush=True,
            )
    raise RuntimeError(
        f"No fully valid three-forest ROI set was found after "
        f"{MAX_RANDOM_ROI_ATTEMPTS} attempts. Last rejection: {last_error}"
    ) from last_error


def refresh_random_forest_data(reason: str = "figure execution") -> dict:
    """Draw a fresh random seed and automatically retry invalid ROIs."""
    global ACTIVE_RANDOM_ROI_SEED, RANDOM_ROI_GENERATOR, forest_data

    ACTIVE_RANDOM_ROI_SEED = int(
        np.random.default_rng().integers(
            0, np.iinfo(np.uint32).max, dtype=np.uint32
        )
    )
    RANDOM_ROI_GENERATOR = np.random.default_rng(ACTIVE_RANDOM_ROI_SEED)
    print(
        f"[NEW RANDOM ROI] reason={reason} | seed={ACTIVE_RANDOM_ROI_SEED}",
        flush=True,
    )
    forest_data = (
        _load_all_forest_rois_with_retry(reason)
        if RUN_QUALITATIVE_FIGURES else {}
    )
    return forest_data


forest_data = (
    _load_all_forest_rois_with_retry("initial qualitative ROI selection")
    if RUN_QUALITATIVE_FIGURES else {}
)

roi_report = pd.DataFrame([
    {
        "forest": forest,
        "year": SITES[forest]["year"],
        "crs": str(data["crs"]),
        "left": data["bounds"][0],
        "bottom": data["bounds"][1],
        "right": data["bounds"][2],
        "top": data["bounds"][3],
        "common_valid_fraction": data["coverage"],
        "available_products": ", ".join(data["maps"]),
        "excluded_low_global_coverage": ", ".join(
            SITES[forest].get("roi_excluded_products", [])
        ),
        "excluded_not_fully_valid_locally": ", ".join(
            SITES[forest].get("locally_invalid_products", [])
        ),
        "roi_selection_status": SITES[forest].get("roi_selection_status"),
        "roi_candidate_key": SITES[forest].get("roi_candidate_key"),
        "roi_history_count": SITES[forest].get("roi_history_count"),
        "random_roi_seed": ACTIVE_RANDOM_ROI_SEED,
        "sentinel2_fallback": str(SITES[forest]["s2"]) if SITES[forest]["s2"] else "NOT FOUND",
        "basemap_source": data["basemap_source"],
        "google_2020_cache": str(_google_cache_path(forest, data["bounds"])),
    }
    for forest, data in forest_data.items()
])
display(roi_report)
roi_report.to_csv(OUT_DIR / "01_selected_roi_audit.csv", index=False)
(OUT_DIR / "01_selected_roi_audit.json").write_text(
    roi_report.to_json(orient="records", indent=2),
    encoding="utf-8",
)


In [ ]:
def square_center_crop(image: np.ndarray | None) -> np.ndarray | None:
    """Crop a satellite image to a centred square without geometric stretching."""
    if image is None:
        return None
    array = np.asarray(image)
    if array.ndim < 2:
        raise ValueError(f"Unexpected basemap shape: {array.shape}")
    height, width = array.shape[:2]
    side = min(height, width)
    row0 = (height - side) // 2
    col0 = (width - side) // 2
    return array[row0:row0 + side, col0:col0 + side, ...]


def save_three_formats(fig, stem: str) -> list[Path]:
    paths = []
    for suffix in ("png", "svg", "pdf"):
        path = OUT_DIR / f"{stem}.{suffix}"
        kwargs = {"bbox_inches": "tight", "facecolor": "white", "pad_inches": 0.02}
        if suffix == "png":
            kwargs["dpi"] = EXPORT_DPI
        fig.savefig(path, **kwargs)
        paths.append(path)
    return paths


def configure_square_panel(ax, image_shape=None):
    """Give every satellite/CHM panel the exact same square drawing box."""
    ax.set_box_aspect(1)
    ax.set_aspect("equal", adjustable="box")
    ax.set_anchor("C")
    ax.margins(0)
    if image_shape is not None:
        height, width = image_shape[:2]
        ax.set_xlim(-0.5, width - 0.5)
        ax.set_ylim(height - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])


def overlay_nodata_crosses(ax, array: np.ndarray, block_pixels: int = 3):
    """Show true NaN/NoData areas as light-grey square cells crossed with an X."""
    invalid = ~np.isfinite(np.asarray(array))
    if not np.any(invalid):
        return

    rows, cols = invalid.shape
    block = max(1, int(block_pixels))
    line_color = "#777777"
    line_width = 0.32

    # When the product is completely unavailable for the displayed site,
    # use the same single crossed-box convention as in the scatter plots.
    if np.all(invalid):
        left, right = -0.5, cols - 0.5
        top, bottom = -0.5, rows - 0.5
        ax.plot(
            [left, right, right, left, left],
            [top, top, bottom, bottom, top],
            color=line_color, linewidth=0.9, zorder=7,
        )
        ax.plot([left, right], [top, bottom], color=line_color, linewidth=0.9, zorder=7)
        ax.plot([left, right], [bottom, top], color=line_color, linewidth=0.9, zorder=7)
        return

    # Draw an X only where the complete display cell is NoData. Valid zero-height
    # pixels are deliberately excluded because zero is a legitimate map value.
    for row0 in range(0, rows, block):
        row1 = min(row0 + block, rows)
        for col0 in range(0, cols, block):
            col1 = min(col0 + block, cols)
            if not np.all(invalid[row0:row1, col0:col1]):
                continue
            left, right = col0 - 0.5, col1 - 0.5
            top, bottom = row0 - 0.5, row1 - 0.5
            ax.plot(
                [left, right, right, left, left],
                [top, top, bottom, bottom, top],
                color=line_color, linewidth=line_width, zorder=7,
            )
            ax.plot([left, right], [top, bottom], color=line_color, linewidth=line_width, zorder=7)
            ax.plot([left, right], [bottom, top], color=line_color, linewidth=line_width, zorder=7)


def add_scale_bar(
    ax,
    pixel_size: float,
    length_metres: float = 200.0,
    subdivisions: int = 2,
):
    """Clear 200 m scale: endpoint labels and an unlabeled 100 m tick."""
    image = ax.images[-1].get_array()
    rows, cols = image.shape[:2]
    width_metres = cols * float(pixel_size)
    if float(length_metres) > 0.42 * width_metres:
        raise RuntimeError(
            f"The {length_metres:.0f} m scale bar is too long for "
            f"a {width_metres:.1f} m displayed extent."
        )

    x0 = 0.06
    x1 = x0 + float(length_metres) / width_metres
    x_mid = 0.5 * (x0 + x1)
    y0 = 0.060
    tick = 0.014

    # Thin dark underlay keeps the scale visible over bright and dark terrain.
    ax.plot(
        [x0, x1], [y0, y0],
        transform=ax.transAxes,
        color="black", lw=3.8,
        solid_capstyle="butt", zorder=20,
    )
    ax.plot(
        [x0, x1], [y0, y0],
        transform=ax.transAxes,
        color="white", lw=2.0,
        solid_capstyle="butt", zorder=21,
    )
    for xpos in (x0, x_mid, x1):
        ax.plot(
            [xpos, xpos], [y0 - tick, y0 + tick],
            transform=ax.transAxes,
            color="black", lw=2.8, zorder=20,
        )
        ax.plot(
            [xpos, xpos], [y0 - tick, y0 + tick],
            transform=ax.transAxes,
            color="white", lw=1.3, zorder=21,
        )

    # Three explicit labels. The last label starts at the final tick, so
    # "100" and "200 m" remain separated even in compact matrix panels.
    for xpos, label, horizontal_alignment in (
        (x0, "0", "left"),
        (x_mid, f"{0.5 * float(length_metres):.0f}", "center"),
        (x1, f"{float(length_metres):.0f} m", "left"),
    ):
        ax.text(
            xpos, y0 + 0.020,
            label,
            transform=ax.transAxes,
            color="white",
            ha=horizontal_alignment, va="bottom",
            fontsize=4.8,
            fontweight="bold",
            path_effects=[
                matplotlib.patheffects.withStroke(
                    linewidth=1.1, foreground="black"
                )
            ],
            zorder=22,
        )

def plot_forest_qualitative(forest: str):
    cfg = SITES[forest]
    data = forest_data[forest]
    products = [name for name in PRODUCT_ORDER if name in data["maps"]]
    panels = ["Satellite context"] + products
    ncols = len(panels)

    fig = plt.figure(figsize=(2.75 * ncols + 0.55, 2.75))
    gs = fig.add_gridspec(
        1, ncols + 1,
        width_ratios=[1.0] * ncols + [0.045],
        wspace=0.018,
    )
    axes = np.array([fig.add_subplot(gs[0, index]) for index in range(ncols)])
    cax = fig.add_subplot(gs[0, -1])
    cmap = mpl.colormaps[CMAP_NAME].copy()
    cmap.set_bad(NODATA_COLOR)
    norm = Normalize(0.0, cfg["vmax"])
    common = data["common_mask"]
    map_image = None

    for index, (ax, panel) in enumerate(zip(axes, panels)):
        if panel == "Satellite context":
            if data["basemap"] is None:
                ax.set_facecolor(NODATA_COLOR)
                ax.text(0.5, 0.5, "Satellite context\nunavailable", ha="center", va="center", transform=ax.transAxes)
            else:
                basemap_square = square_center_crop(data["basemap"])
                ax.imshow(
                    basemap_square,
                    interpolation="bilinear",
                    aspect="equal",
                )
            title = "Google Maps"
            # Publication lock: every forest displays the same nominal
            # 700 m square ROI, so a 200 m bar occupies exactly 200/700
            # of the panel width, independent of basemap pixel dimensions.
            displayed_width_metres = float(cfg["roi_metres"])
            context_pixel_size = (
                displayed_width_metres / basemap_square.shape[1]
                if data["basemap"] is not None else data["pixel_size"]
            )
            add_scale_bar(
                ax,
                context_pixel_size,
                length_metres=200.0,
                subdivisions=2,
            )
            ax.text(
                0.975, 0.025, str(BASEMAP_REFERENCE_YEAR),
                transform=ax.transAxes,
                ha="right", va="bottom",
                color="white", fontsize=7.0,
                fontweight="semibold", zorder=22,
            )
        else:
            shown = data["maps"][panel]
            map_image = ax.imshow(shown, cmap=cmap, norm=norm, interpolation="nearest")
            overlay_nodata_crosses(ax, shown)
            title = PRODUCT_TITLES[panel]

        ax.set_title(title, pad=4)
        plotted_shape = (
            basemap_square.shape
            if panel == "Satellite context" and data["basemap"] is not None
            else (
                data["maps"][panel].shape
                if panel in data["maps"] else None
            )
        )
        configure_square_panel(ax, plotted_shape)

    cbar = fig.colorbar(map_image, cax=cax)
    cbar.set_label("Canopy height (m)")
    cbar.set_ticks(
            [0, 10, 20, 30, 40]
            if cfg["vmax"] == 40.0
            else [0, 5, 10, 15, 20]
        )
    cbar.ax.yaxis.set_major_formatter(
        mpl.ticker.FuncFormatter(lambda value, _: f"{value:g}")
    )
    paths = save_three_formats(fig, f"Fig_qualitative_{forest.lower()}_random_full_valid_v6_aligned")
    plt.show()
    plt.close(fig)
    print("\n".join(f"Saved: {path}" for path in paths))
    return paths



if RUN_QUALITATIVE_FIGURES:
    refresh_random_forest_data("per-forest qualitative figures")

forest_figure_paths = ({
    forest: plot_forest_qualitative(forest)
    for forest in SITES
} if RUN_QUALITATIVE_FIGURES else {})


In [ ]:
def plot_combined_matrix(realization_index: int = 1):
    forests = list(SITES)
    columns = ["Satellite context"] + PRODUCT_ORDER
    fig = plt.figure(figsize=(15.4, 7.7))
    gs = fig.add_gridspec(
        len(forests),
        len(columns) + 1,
        width_ratios=[1] * len(columns) + [0.045],
        wspace=0.018,
        hspace=0.025,
    )

    for row, forest in enumerate(forests):
        cfg = SITES[forest]
        data = forest_data[forest]
        common = data["common_mask"]
        cmap = mpl.colormaps[CMAP_NAME].copy()
        cmap.set_bad(NODATA_COLOR)
        norm = Normalize(0.0, cfg["vmax"])
        row_mappable = None

        for col, panel in enumerate(columns):
            ax = fig.add_subplot(gs[row, col])
            if panel == "Satellite context":
                if data["basemap"] is None:
                    ax.set_facecolor(NODATA_COLOR)
                    ax.text(0.5, 0.5, "Not available", ha="center", va="center", transform=ax.transAxes)
                else:
                    basemap_square = square_center_crop(data["basemap"])
                    ax.imshow(
                        basemap_square,
                        interpolation="bilinear",
                        aspect="equal",
                    )
                # Publication lock: every forest displays the same nominal
                # 700 m square ROI, so a 200 m bar occupies exactly 200/700
                # of the panel width, independent of basemap pixel dimensions.
                displayed_width_metres = float(cfg["roi_metres"])
                context_pixel_size = (
                    displayed_width_metres / basemap_square.shape[1]
                    if data["basemap"] is not None else data["pixel_size"]
                )
                add_scale_bar(
                    ax,
                    context_pixel_size,
                    length_metres=200.0,
                    subdivisions=2,
                )
                ax.text(
                    0.975, 0.025, str(BASEMAP_REFERENCE_YEAR),
                    transform=ax.transAxes,
                    ha="right", va="bottom",
                    color="white", fontsize=7.0,
                    fontweight="semibold", zorder=22,
                )
            elif panel in data["maps"]:
                shown = data["maps"][panel]
                row_mappable = ax.imshow(shown, cmap=cmap, norm=norm, interpolation="nearest")
                overlay_nodata_crosses(ax, shown)
            else:
                ax.set_facecolor(NODATA_COLOR)
                ax.text(0.5, 0.5, "Not available", ha="center", va="center", transform=ax.transAxes, color="0.35")

            if row == 0:
                ax.set_title(
                    (
"Google Maps"
                    )
                    if panel == "Satellite context" else PRODUCT_TITLES[panel],
                    pad=4,
                )
            if col == 0:
                ax.set_ylabel(f"{cfg['ecosystem']}\n({forest} Forest)", fontweight="semibold")
            plotted_shape = (
                basemap_square.shape
                if panel == "Satellite context" and data["basemap"] is not None
                else (
                    data["maps"][panel].shape
                    if panel in data["maps"] else None
                )
            )
            configure_square_panel(ax, plotted_shape)

        cax = fig.add_subplot(gs[row, -1])
        cbar = fig.colorbar(row_mappable, cax=cax)
        # Inset each row colorbar vertically so adjacent endpoint labels
        # never overlap, while the CHM panels themselves remain contiguous.
        cax_position = cax.get_position()
        cax.set_position([
            cax_position.x0,
            cax_position.y0 + 0.035 * cax_position.height,
            cax_position.width,
            0.93 * cax_position.height,
        ])
        cbar.set_label("Canopy height (m)", fontsize=8)
        cbar.set_ticks(
            [0, 10, 20, 30, 40]
            if cfg["vmax"] == 40.0
            else [0, 5, 10, 15, 20]
        )
        cbar.ax.yaxis.set_major_formatter(
            mpl.ticker.FuncFormatter(lambda value, _: f"{value:g}")
        )
        cbar.ax.tick_params(labelsize=7, pad=2)

    output_stem = (
        "Fig_qualitative_three_forests_matrix_random_full_valid_v6_aligned"
        f"_R{realization_index:02d}_seed{ACTIVE_RANDOM_ROI_SEED}"
    )
    paths = save_three_formats(fig, output_stem)
    plt.show()
    plt.close(fig)
    print("\n".join(f"Saved: {path}" for path in paths))
    return paths



N_RANDOM_COMBINED_MATRICES = 5

combined_figure_paths = []
if RUN_QUALITATIVE_FIGURES:
    for realization_index in range(1, N_RANDOM_COMBINED_MATRICES + 1):
        refresh_random_forest_data(
            f"combined qualitative matrix {realization_index}/"
            f"{N_RANDOM_COMBINED_MATRICES}"
        )
        combined_figure_paths.extend(
            plot_combined_matrix(realization_index=realization_index)
        )

print(
    f"Generated {len(combined_figure_paths)} independent random matrix/matrices."
)


In [ ]:
# AUTONOMOUS_FIGURE_MANIFEST_V2
# Safe in a fresh kernel: this cell inventories real files already exported on disk.
from pathlib import Path
import pandas as pd
from IPython.display import display

_project = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
_out_dir = globals().get(
    "OUT_DIR",
    _project / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Inference_Maps",
)
_out_dir = Path(_out_dir)
_out_dir.mkdir(parents=True, exist_ok=True)


def _flatten_paths(value):
    if value is None:
        return []
    if isinstance(value, (str, Path)):
        return [Path(value)]
    paths = []
    for item in value:
        paths.extend(_flatten_paths(item))
    return paths


manifest_rows = []
_forest_paths = globals().get("forest_figure_paths", {}) or {}
_forest_data = globals().get("forest_data", {}) or {}
for forest, paths in _forest_paths.items():
    source = (_forest_data.get(forest, {}) or {}).get("basemap_source", "not recorded in this kernel")
    for path in _flatten_paths(paths):
        if path.is_file():
            manifest_rows.append({
                "figure": f"qualitative_{str(forest).lower()}",
                "forest": str(forest),
                "format": path.suffix.lower().lstrip("."),
                "path": str(path.resolve()),
                "basemap_source": source,
            })

_combined_paths = _flatten_paths(globals().get("combined_figure_paths", []))
_sites = list((globals().get("SITES", {}) or {}).keys())
_combined_source = "; ".join(
    f"{forest}: {(_forest_data.get(forest, {}) or {}).get('basemap_source', 'not recorded in this kernel')}"
    for forest in _sites
) or "not recorded in this kernel"
for path in _combined_paths:
    if path.is_file():
        manifest_rows.append({
            "figure": "qualitative_three_forests_matrix",
            "forest": "All",
            "format": path.suffix.lower().lstrip("."),
            "path": str(path.resolve()),
            "basemap_source": _combined_source,
        })

# In a fresh kernel, recover every real exported figure instead of failing on
# variables created by optional plotting cells. No inference or plotting is rerun.
if not manifest_rows:
    for path in sorted(_out_dir.rglob("*")):
        if not path.is_file() or path.suffix.lower() not in {".png", ".pdf", ".svg"}:
            continue
        name = path.stem.lower()
        if "qualitative" in name or "piw90" in name or "uncertainty" in name:
            forest = next((f for f in ("Ifran", "Maamoura", "Agadir") if f.lower() in name), "All")
            figure = (
                "our_model_global_piw90_companion" if "our_model" in name and "piw90" in name
                else "global_piw90_companion" if "piw90" in name
                else "qualitative_three_forests_matrix" if forest == "All"
                else f"qualitative_{forest.lower()}"
            )
            manifest_rows.append({
                "figure": figure,
                "forest": forest,
                "format": path.suffix.lower().lstrip("."),
                "path": str(path.resolve()),
                "basemap_source": "read from existing exported artifact",
            })

manifest = pd.DataFrame(
    manifest_rows,
    columns=["figure", "forest", "format", "path", "basemap_source"],
).drop_duplicates(subset=["path"]).sort_values(
    ["figure", "forest", "format"], ignore_index=True
)
manifest_path = _out_dir / "figure_manifest.csv"
manifest.to_csv(manifest_path, index=False)
display(manifest)
print(f"Manifest saved: {manifest_path}")
print(f"[PASS] {len(manifest)} existing figure artifact(s) inventoried; no inference was rerun.")
